### Purpose
This notebook does the following:
1. Reads data from bronze.
2. Cleans data and removes duplicates.
3. Adapts data format when needed (for example, renaming a column).
4. Saves processed data in a Delta table and invalid data in a separate Delta table for analyst review.
5. Checks history and maintains Delta tables.

In [ ]:
# Get notebook parameters from the Azure Data Factory pipeline
dbutils.widgets.text("_pipeline_run_id","0478ce36-b895-48a0-8a08-1b10430247ca")
dbutils.widgets.text("_processing_date","21-05-2024")
dbutils.widgets.text("_account_name","datalakexxxxx")
_pipeline_run_id = dbutils.widgets.get("_pipeline_run_id")
bronze_processing_date = dbutils.widgets.get("_processing_date")
accountName = dbutils.widgets.get("_account_name")
print(_pipeline_run_id)
print(accountName)
print(bronze_processing_date)

In [ ]:
# Unity Catalog external locations handle ADLS authentication via the Access Connector managed identity.
# No account key or secret scope is needed — just the storage account name (passed as a notebook parameter).

In [ ]:
# Define the location of my files
bronzeSource = f'abfss://bronze@{accountName}.dfs.core.windows.net/nybabynames'
silverTarget = f'abfss://silver@{accountName}.dfs.core.windows.net/nybabynames'
silverErrors = f'abfss://silver@{accountName}.dfs.core.windows.net/nybabynameserrors'

bronze_table_name =  "bronze.new_york_baby_names"
silver_table_name =  "silver.new_york_baby_names"
silver_errors_table_name =  "silver.new_york_baby_names_errors"

In [ ]:
# Read data from Data Lake
from pyspark.sql.functions import to_date, lit, col
import json

# Retrieve only the data processed for the requested date
condition = to_date(col("_processing_date")) == to_date(lit(bronze_processing_date), "dd-MM-yyyy")
gridDataBronze = spark.read.table(bronze_table_name).filter(condition)

if gridDataBronze.limit(1).count() == 0:
    dbutils.notebook.exit(json.dumps({"status": "no_data", "table": silver_table_name, "message": "No rows found for the requested processing date"}))

gridDataBronze.printSchema()



In [ ]:
# Correct data structure and add metadata

# 1. Rename name_count column to count
gridDataBronze = gridDataBronze.withColumnRenamed("name_count", "count")

gridDataBronze.printSchema()

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

# Data quality
gridCleanDF = gridDataBronze.filter("year IS NOT NULL AND first_name IS NOT NULL AND county IS NOT NULL AND sex IS NOT NULL AND count IS NOT NULL AND count > 0")

# Data deduplication
# Partition by business keys and keep the latest record per group based on file modification timestamp.
gridDataWindowSpec = Window.partitionBy("year", "first_name", "county", "sex").orderBy(col("_input_file_modification_date").desc(), "count")
findLatestDF = gridCleanDF.withColumn("row_number", row_number().over(gridDataWindowSpec)).filter("row_number == 1").drop("row_number")

# Wrong data detected
gridDataErrorDF = gridDataBronze.subtract(findLatestDF)

gridDataDf = findLatestDF

In [ ]:
# Save invalid data in a Delta table for analyst review
from delta.tables import DeltaTable

# Check if the errors path already contains a Delta table
if DeltaTable.isDeltaTable(spark, silverErrors):
    # If yes, append data to the existing Delta table
    gridDataErrorDF.write.mode("append").format("delta").save(silverErrors)
else:

    # If no, save the data
    gridDataErrorDF.write.mode("overwrite").format("delta").save(silverErrors)

In [ ]:
from delta.tables import DeltaTable

# Check whether the silver path already contains a Delta table
# Databricks provides a null-safe equal operator (<=>). It returns False when only one operand is NULL and True when both operands are NULL.
if DeltaTable.isDeltaTable(spark, silverTarget):

    # If yes, merge data with the existing Delta table
    DeltaTable.forPath(spark, silverTarget).alias("target").merge(
        source = gridDataDf.alias("src"),
        condition = "target.year <=> src.year and target.first_name <=> src.first_name and target.county <=> src.county and target.sex <=> src.sex"
    ).whenMatchedUpdate(
        condition = "target._input_file_modification_date < src._input_file_modification_date",
        set = {
            "count" : "src.count",
            "_processing_date" : "src._processing_date",
            "_pipeline_run_id" : "src._pipeline_run_id",
            "_input_filename" : "src._input_filename",
            "_input_file_modification_date" : "src._input_file_modification_date"
        }
    ).whenNotMatchedInsertAll().execute()
else:

    # If no, save the data to silver
    gridDataDf.write.mode("overwrite").format("delta").save(silverTarget)

In [ ]:
# Create the schema and table, if required

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {silver_table_name} USING delta LOCATION '{silverTarget}'")

# Note: spark.sql is used here to inject the silver path with an f-string.

In [ ]:
%sql
-- This is not necessary from a pipeline perspective; it involves checking table information as a learning experience.

DESCRIBE EXTENDED silver.new_york_baby_names

-- Location: stored in the storage account
-- Provider (format): Delta

In [ ]:
%sql
-- This is not necessary from a pipeline perspective; it involves showing the transaction log on the delta version as a learning experience.

SELECT version, operationMetrics, operationMetrics.numOutputRows, operationMetrics.numTargetRowsInserted, operationMetrics.numTargetRowsUpdated, operationMetrics.numTargetRowsDeleted
FROM (DESCRIBE HISTORY silver.new_york_baby_names)

In [ ]:
%sql

-- Check your result for testing. Do not do this in production!
-- SELECT first_name, sum(count) as cnt
-- FROM silver.new_york_baby_names
-- GROUP BY (first_name)
-- ORDER BY cnt DESC
-- LIMIT 10





In [ ]:
# Maintenance for Delta tables

# To optimize Delta table performance, we run two commands:
# 1. optimize(): compacts small files.
# 2. vacuum(): removes old files. This reduces overhead but limits time travel.

# Databricks recommends frequently running OPTIMIZE to compact small files.
# This operation does not remove old files. To remove them, run VACUUM (https://learn.microsoft.com/en-us/azure/databricks/delta/vacuum).
# https://learn.microsoft.com/en-us/azure/databricks/delta/best-practices#--compact-files

# In Azure, predictive optimization can be used (https://learn.microsoft.com/en-us/azure/databricks/optimizations/predictive-optimization#what-operations-does-predictive-optimization-run).
# It has prerequisites, such as Premium plan and managed tables (https://learn.microsoft.com/en-us/azure/databricks/optimizations/predictive-optimization#prerequisites-for-predictive-optimization).

gridDataDelta = DeltaTable.forName(spark, silver_table_name)

# In this example, we run optimize and vacuum every 30 days
if gridDataDelta.history(30).filter("operation = 'VACUUM START'").count() == 0:
    gridDataDelta.optimize()
    gridDataDelta.vacuum() # default = 7 days

if spark.catalog.tableExists(silver_errors_table_name):
    gridDataDelta = DeltaTable.forName(spark, silver_errors_table_name)
    # In this example, we run optimize and vacuum every 30 days
    if gridDataDelta.history(30).filter("operation = 'VACUUM START'").count() == 0:
        gridDataDelta.optimize()
        gridDataDelta.vacuum() # default = 7 days

import json
dbutils.notebook.exit(json.dumps({"status": "success", "table": silver_table_name}))